<a href="https://colab.research.google.com/github/evinracher/3008410-intelligent-systems/blob/main/week6/exercise2/adversarial_text_attacks_and_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adversarial attacks

Adversarial attacks on natural language processing models aim to expose vulnerabilities by subtly altering input texts to mislead the model’s predictions. Various attack recipes have been developed, each with unique strategies ranging from word-level synonym replacements to character-level perturbations. The table below compares some of the most popular TextAttack recipes, highlighting their key features, strengths, and typical use cases.


| Attack Name       | Description                                                | Strengths                              | Weaknesses                         | Typical Use Case                   |
|-------------------|------------------------------------------------------------|--------------------------------------|-----------------------------------|----------------------------------|
| PWWSRen2019       | Probability Weighted Word Saliency attack. Replaces important words weighted by model sensitivity. | Effective at word-level perturbations, fast | Can struggle with complex context | Text classification adversarial testing |
| TextFoolerJin2019 | Uses word embeddings to replace words with synonyms preserving semantics. | Preserves semantic meaning, intuitive | Sometimes produces unnatural sentences | Robust synonym-based attacks     |
| DeepWordBug       | Generates character-level perturbations like typos and swaps to fool models. | Effective against typo-sensitive models | Less effective on robust models   | Testing typo robustness          |
| BAEGarg2019       | Generates adversarial examples by masking and replacing words using BERT predictions. | Context-aware replacements            | Computationally expensive          | Semantic-aware adversarial attacks |
| HotFlip           | Uses gradient information to flip characters for adversarial attacks. | Gradient-guided, effective character-level attacks | Needs white-box access to gradients | White-box adversarial testing    |
| TextBugger        | Combines character and word-level perturbations, including typos and synonym replacements. | Versatile, black-box attacks          | May reduce readability             | Black-box adversarial generation |

---

| Attack Name       | Original Text | Adversarial Example | Key Change |
|-------------------|--------------|--------------------|-----------|
| PWWSRen2019 | The movie was absolutely wonderful and I loved every minute of it. | The movie was absolutely **ordinary** and I loved every minute of it. | wonderful → ordinary |
| TextFoolerJin2019 | The movie was absolutely wonderful and I loved every minute of it. | The movie was absolutely **marvelous** and I **adored** every minute of it. | wonderful → marvelous, loved → adored |
| DeepWordBug | The movie was absolutely wonderful and I loved every minute of it. | The **mov1e** was **absolut3ly wond3rful** and I loved every minute of it. | character typos |


In [1]:
!pip install \
    textattack==0.3.10 \
    transformers==4.44.2 \
    tokenizers==0.19.1 \
    datasets==3.6.0 \
    evaluate==0.4.3 \
    sentence-transformers==3.3.1 \
    textblob==0.18.0 \
    torch>=2.0.0 \
    --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


# Login to Hugging Face
This notebook uses public models and datasets, so login is not required.
If you hit rate limits while downloading, you can authenticate with a token.

Options:
- Set an environment variable `HF_TOKEN`.
- (Kaggle) Provide a JSON file at `/kaggle/input/autenti/AUTH nn.json` with an `API_KEY` field.


In [2]:
import json
import os
from pathlib import Path

from huggingface_hub import login

# Optional: authenticate to increase rate limits when downloading models/datasets.
# For the public assets used here (GLUE SST-2 and DistilBERT SST-2), login is not required.

token = os.getenv("HF_TOKEN")

# Kaggle-specific fallback: allow reading the token from a mounted dataset, if present.
config_path = Path("/kaggle/input/datasets/reinaldolopeznarvaez/api-key/AUTH.json")
if token is None and config_path.exists():
    with config_path.open("r", encoding="utf-8") as f:
        config = json.load(f)
    token = config.get("API_KEY")

if token:
    login(token=token)
    print("Successful login to Hugging Face.")
else:
    print("No Hugging Face token found; continuing without login.")


No Hugging Face token found; continuing without login.


In [3]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face using Colab HF_TOKEN.")
except userdata.SecretNotFoundError:
    print("Colab secret 'HF_TOKEN' not found. Please add it to your Colab secrets.")
except Exception as e:
    print(f"An error occurred during Hugging Face login: {e}")

Successfully logged into Hugging Face using Colab HF_TOKEN.


# Import libraries

In [4]:
import logging
import warnings

import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    logging as hf_logging,
)

from textattack.attack_recipes import PWWSRen2019
from textattack.models.wrappers import HuggingFaceModelWrapper

# Optional (not used in the baseline code below): semantic-similarity-based signals for detection.
from sentence_transformers import SentenceTransformer, util

from textblob import TextBlob

# Print versions to make runs reproducible when sharing results.
import transformers
import datasets
import evaluate
import textattack
import sentence_transformers
import textblob


# Reduce noise in notebook output.
warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


textattack: Updating TextAttack package dependencies.
textattack: Downloading NLTK required packages.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw to /root/nltk_data...
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape s

device: cuda


# Adversarial Functions

In [5]:
def preprocess_text(text: str) -> str:
    """Optional normalization step.

    NOTE: `TextBlob(text).correct()` is slow and can change semantics; keep it off unless you are
    explicitly studying spelling-correction as a defense.
    """

    return str(TextBlob(text).correct())


def generate_adversarial_examples(model_wrapper, dataset, num_examples: int = 20, verbose: bool = True):
    """Generate adversarial examples with TextAttack.

    Returns a list of dicts with keys:
    - `text`: perturbed text (or original text if the attack fails)
    - `label`: ground-truth label from the original example

    Guardrails:
    - TextAttack can emit failed/skipped results; we keep the original text so fine-tuning can proceed.
    """

    from textattack import AttackArgs, Attacker

    attack = PWWSRen2019.build(model_wrapper)
    attack_args = AttackArgs(
        num_examples=num_examples,
        shuffle=True,        # randomize which samples are attacked
        disable_stdout=True, # silence TextAttack internal prints (our prints still show)
    )
    attacker = Attacker(attack, dataset, attack_args)

    results = []
    for i, result in enumerate(attacker.attack_dataset()):
        # TextAttack results differ for success/fail/skip; guard against missing fields.
        try:
            original_text = result.original_text()
        except Exception:
            original_text = None

        try:
            perturbed_text = result.perturbed_text()
        except Exception:
            perturbed_text = None

        original_label = getattr(getattr(result, "original_result", None), "ground_truth_output", None)
        if original_label is None:
            raise ValueError("Missing ground-truth label; check that the TextAttack dataset includes labels.")

        predicted_label = getattr(getattr(result, "perturbed_result", None), "output", None)

        # If the attack fails, fall back to the original text to avoid downstream crashes.
        attack_text = perturbed_text or original_text
        if attack_text is None:
            continue

        if verbose:
            print(f"\nExample {i + 1}")
            print("Original text:", original_text)
            print("Perturbed text:", perturbed_text)
            print("Ground-truth label:", original_label)
            print("Predicted label after attack:", predicted_label)
            print("Attack successful:", original_label != predicted_label)

        results.append({
            "text": attack_text,
            "label": int(original_label),
        })

    return results


class CustomDataset(torch.utils.data.Dataset):
    """Minimal dataset wrapper for Hugging Face `Trainer`.

    Each item returns tokenized tensors + an integer label.
    """

    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]

        # Fixed-length padding simplifies batching but can waste compute; tune `max_length` as needed.
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


# Datasets
| Dataset Task | Description                                  | # Train Examples | # Validation Examples | # Test Examples     | Labels                                         |
|--------------|----------------------------------------------|------------------|-----------------------|---------------------|------------------------------------------------|
| SST-2        | Sentiment classification (positive/negative) | 67,349           | 872                   | 1,821               | 0 = negative, 1 = positive                     |
| MRPC         | Paraphrase detection                         | 3,668            | 408                   | 1,725               | 0 = not paraphrase, 1 = paraphrase             |
| QQP          | Quora Question Pairs (paraphrase detection) | 363,849          | 40,431                | 390,965             | 0 = not duplicate, 1 = duplicate               |
| QNLI         | Question-answer entailment                   | 104,743          | 5,463                 | 5,463               | 0 = not entailment, 1 = entailment             |
| MNLI         | Multi-genre Natural Language Inference      | 392,702          | 9,815 (matched)        | 9,796 (mismatched)  | 0 = contradiction, 1 = neutral, 2 = entailment |
| CoLA         | Acceptability of English sentences           | 8,551            | 1,043                 | 1,063               | 0 = unacceptable, 1 = acceptable               |
| RTE          | Recognizing Textual Entailment               | 2,490            | 277                   | 3,000               | 0 = not entailment, 1 = entailment             |
| WNLI         | Winograd Schema Challenge                    | 635              | 71                    | 146                 | 0 or 1 (coreference resolution)                |

# Models

| Feature                          | `distilbert-base-uncased-finetuned-sst-2-english`       | `cardiffnlp/twitter-roberta-base-sentiment-latest`          |
|----------------------------------|-----------------------------------------------------------|-------------------------------------------------------------|
| **Architecture**                | DistilBERT (lightweight BERT)                            | RoBERTa (Robustly optimized BERT approach)                  |
| **Pretraining Corpus**          | BooksCorpus + English Wikipedia (via BERT)               | 124M English Tweets                                         |
| **Fine-tuned On**              | SST-2 (Stanford Sentiment Treebank)                      | TweetEval sentiment task                                    |
| **Sentiment Labels**            | 0 = Negative, 1 = Positive                               | 0 = Negative, 1 = Neutral, 2 = Positive                     |
| **Domain Focus**                | General (Movie reviews, formal English)                 | Social media (Twitter-specific)                            |
| **Model Size**                  | ~66M parameters                                          | ~125M parameters                                            |
| **Tokenizer**                   | `distilbert-base-uncased` tokenizer                     | `twitter-roberta-base` tokenizer (handles hashtags, emojis)|
| **Performance (General Text)**  | Good general sentiment classification                   | Weaker on non-social media text                            |
| **Performance (Tweets)**        | Moderate, not optimized for tweets                      | Very strong—trained on tweets                              |
| **Use Case Fit**                | Academic, reviews, formal text                          | Twitter, social listening, short informal text             |


In [6]:
def main():
    # Configuration knobs for the Activities section.
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"

    # How many examples to attack (also equals the number of adversarial examples we attempt to generate).
    attack_examples = 20

    # How many clean examples to include for adversarial training.
    clean_train_examples = 500

    # Validation subset size for quick iteration.
    val_examples = 100

    # Fine-tuning epochs (increase for Activity 2).
    num_train_epochs = 5

    print("Loading model and dataset...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

    # GLUE SST-2: binary sentiment classification.
    hf_dataset = load_dataset("glue", "sst2")
    train_raw = hf_dataset["train"]
    val_raw = hf_dataset["validation"]

    # TextAttack expects a list of (text, label) pairs.
    from textattack.datasets import Dataset

    sample_for_attack = list(zip(
        train_raw["sentence"][:attack_examples],
        train_raw["label"][:attack_examples],
    ))
    textattack_dataset = Dataset(sample_for_attack)

    print("Generating adversarial examples...")
    adv_examples = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )

    print(f"{len(adv_examples)} adversarial examples generated.")

    # Combine clean and adversarial samples for adversarial training.
    clean_data = [
        {"text": x, "label": y}
        for x, y in zip(
            train_raw["sentence"][:clean_train_examples],
            train_raw["label"][:clean_train_examples],
        )
    ]
    combined_data = clean_data + adv_examples

    train_dataset = CustomDataset(combined_data, tokenizer)
    val_dataset = CustomDataset(
        [
            {"text": x, "label": y}
            for x, y in zip(
                val_raw["sentence"][:val_examples],
                val_raw["label"][:val_examples],
            )
        ],
        tokenizer,
    )

    print("Starting fine-tuning...")

    args = TrainingArguments(
        output_dir="./defended_model",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        logging_dir="./logs",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    trainer.train()

    # Persist the defended model locally.
    model.save_pretrained("./defended_model")
    tokenizer.save_pretrained("./defended_model")
    print("Fine-tuned model saved to ./defended_model.")

    # Re-wrap the fine-tuned model and rerun the attack to see whether robustness improved.
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
    _ = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )


if __name__ == "__main__":
    main()


Loading model and dataset...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...



Example 1
Original text: with his usual intelligence and subtlety 
Perturbed text: with his usual tidings and subtlety 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 2
Original text: on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 
Perturbed text: on the forged revenge-of-the-nerds clichés the filmmakers could dredge up 
Ground-truth label: 0
Predicted label after attack: 0
Attack successful: False

Example 3
Original text: remains utterly satisfied to remain the same throughout 
Perturbed text: remains perfectly satisfied to remain the same throughout 
Ground-truth label: 0
Predicted label after attack: 1
Attack successful: True

Example 4
Original text: that 's far too tragic to merit such superficial treatment 
Perturbed text: that 's ALIR too tragical to deservingness such trivial intervention 
Ground-truth label: 0
Predicted label after attack: 0
Attack successful: False

Example 5
Original text: for those moviego

[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Fine-tuned model saved to ./defended_model.
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 9 / 11 / 0 / 20: 100%|██████████| 20/20 [00:02<00:00,  6.96it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 9      |
| Number of failed attacks:     | 11     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 55.0%  |
| Attack success rate:          | 45.0%  |
| Average perturbed word %:     | 40.05% |
| Average num. words per input: | 8.7    |
| Avg num queries:              | 86.95  |
+-------------------------------+--------+

Example 1
Original text: with his usual intelligence and subtlety 
Perturbed text: with his usual word and refinement 
Ground-truth label: 1
Predicted label after attack: 1
Attack successful: False

Example 2
Original text: on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 
Perturbed text: on the whip revenge-of-the-nerds clichés the filmmakers could dredge up 
Ground-truth label: 0
Predicted label after attack: 

# Activities

1. Increase the number of examples. What are your conclusions about the model's performance? Does the model improve or get worse? *Hint: Example Quantity*

2. Increase the number of training epochs. Is there any improvement in the model?

3.  Change the attacker to DeepWordBugGao2018. What are the main differences compared to PWWSRen2019? Which attacker is more difficult to correct?

4. Change the model to cardiffnlp/twitter-roberta-base-sentiment-latest. What is the performance? Write your conclusions about the whole process.

## Solution 1

In [7]:

def main():
    # Configuration knobs for the Activities section.
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"

    # How many examples to attack (also equals the number of adversarial examples we attempt to generate).
    attack_examples = 50

    # How many clean examples to include for adversarial training.
    clean_train_examples = 500

    # Validation subset size for quick iteration.
    val_examples = 100

    # Fine-tuning epochs (increase for Activity 2).
    num_train_epochs = 5

    print("Loading model and dataset...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

    # GLUE SST-2: binary sentiment classification.
    hf_dataset = load_dataset("glue", "sst2")
    train_raw = hf_dataset["train"]
    val_raw = hf_dataset["validation"]

    # TextAttack expects a list of (text, label) pairs.
    from textattack.datasets import Dataset

    sample_for_attack = list(zip(
        train_raw["sentence"][:attack_examples],
        train_raw["label"][:attack_examples],
    ))
    textattack_dataset = Dataset(sample_for_attack)

    print("Generating adversarial examples...")
    adv_examples = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )

    print(f"{len(adv_examples)} adversarial examples generated.")

    # Combine clean and adversarial samples for adversarial training.
    clean_data = [
        {"text": x, "label": y}
        for x, y in zip(
            train_raw["sentence"][:clean_train_examples],
            train_raw["label"][:clean_train_examples],
        )
    ]
    combined_data = clean_data + adv_examples

    train_dataset = CustomDataset(combined_data, tokenizer)
    val_dataset = CustomDataset(
        [
            {"text": x, "label": y}
            for x, y in zip(
                val_raw["sentence"][:val_examples],
                val_raw["label"][:val_examples],
            )
        ],
        tokenizer,
    )

    print("Starting fine-tuning...")

    args = TrainingArguments(
        output_dir="./defended_model",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        logging_dir="./logs",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    trainer.train()

    # Persist the defended model locally.
    model.save_pretrained("./defended_model")
    tokenizer.save_pretrained("./defended_model")
    print("Fine-tuned model saved to ./defended_model.")

    # Re-wrap the fine-tuned model and rerun the attack to see whether robustness improved.
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
    _ = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )


if __name__ == "__main__":
    main()


Loading model and dataset...


[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Generating adversarial examples...
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 37 / 12 / 1 / 50: 100%|██████████| 50/50 [00:05<00:00,  8.55it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 37     |
| Number of failed attacks:     | 12     |
| Number of skipped attacks:    | 1      |
| Original accuracy:            | 98.0%  |
| Accuracy under attack:        | 24.0%  |
| Attack success rate:          | 75.51% |
| Average perturbed word %:     | 27.43% |
| Average num. words per input: | 9.04   |
| Avg num queries:              | 75.12  |
+-------------------------------+--------+



Example 1
Original text: hide new secretions from the parental units 
Perturbed text: enshroud Modern secretions from the parental units 
Ground-truth label: 0
Predicted label after attack: 1
Attack successful: True

Example 2
Original text: cross swords with the best of them and 
Perturbed text: thwart swords with the best of them and 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 3
Original text: are more deeply thought through than in most ` right-thinking ' films 
Perturbed text: are more deeply mean through than in most ` right-thinking ' films 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 4
Original text: very good viewing alternative 
Perturbed text: very unspoilt viewing alternative 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 5
Original text: equals the original and in some ways even betters it 
Perturbed text: equals the original and in some ways eve

[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Fine-tuned model saved to ./defended_model.
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 23 / 27 / 0 / 50: 100%|██████████| 50/50 [00:08<00:00,  6.12it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 23     |
| Number of failed attacks:     | 27     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 54.0%  |
| Attack success rate:          | 46.0%  |
| Average perturbed word %:     | 44.0%  |
| Average num. words per input: | 9.04   |
| Avg num queries:              | 95.72  |
+-------------------------------+--------+

Example 1
Original text: hide new secretions from the parental units 
Perturbed text: skin freshly secretions from the parental whole 
Ground-truth label: 0
Predicted label after attack: 1
Attack successful: True

Example 2
Original text: cross swords with the best of them and 
Perturbed text: grumpy swords with the outflank of them and 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 3
Original t

### Answer point 1

We see that the improvement doesn't change a lot. The number of the examples or queries made to test doesn't affect improvement of the model

## Solution 2

In [9]:
# Solution 2
def main():
    # Configuration knobs for the Activities section.
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"

    # How many examples to attack (also equals the number of adversarial examples we attempt to generate).
    attack_examples = 20

    # How many clean examples to include for adversarial training.
    clean_train_examples = 500

    # Validation subset size for quick iteration.
    val_examples = 100

    # Fine-tuning epochs (increase for Activity 2).
    num_train_epochs = 15

    print("Loading model and dataset...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

    # GLUE SST-2: binary sentiment classification.
    hf_dataset = load_dataset("glue", "sst2")
    train_raw = hf_dataset["train"]
    val_raw = hf_dataset["validation"]

    # TextAttack expects a list of (text, label) pairs.
    from textattack.datasets import Dataset

    sample_for_attack = list(zip(
        train_raw["sentence"][:attack_examples],
        train_raw["label"][:attack_examples],
    ))
    textattack_dataset = Dataset(sample_for_attack)

    print("Generating adversarial examples...")
    adv_examples = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=False,
    )

    print(f"{len(adv_examples)} adversarial examples generated.")

    # Combine clean and adversarial samples for adversarial training.
    clean_data = [
        {"text": x, "label": y}
        for x, y in zip(
            train_raw["sentence"][:clean_train_examples],
            train_raw["label"][:clean_train_examples],
        )
    ]
    combined_data = clean_data + adv_examples

    train_dataset = CustomDataset(combined_data, tokenizer)
    val_dataset = CustomDataset(
        [
            {"text": x, "label": y}
            for x, y in zip(
                val_raw["sentence"][:val_examples],
                val_raw["label"][:val_examples],
            )
        ],
        tokenizer,
    )

    print("Starting fine-tuning...")

    args = TrainingArguments(
        output_dir="./defended_model",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        logging_dir="./logs",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    trainer.train()

    # Persist the defended model locally.
    model.save_pretrained("./defended_model")
    tokenizer.save_pretrained("./defended_model")
    print("Fine-tuned model saved to ./defended_model.")

    # Re-wrap the fine-tuned model and rerun the attack to see whether robustness improved.
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
    _ = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=False,
    )


if __name__ == "__main__":
    main()


Loading model and dataset...


[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Generating adversarial examples...
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 15 / 4 / 1 / 20: 100%|██████████| 20/20 [00:02<00:00,  9.01it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 15     |
| Number of failed attacks:     | 4      |
| Number of skipped attacks:    | 1      |
| Original accuracy:            | 95.0%  |
| Accuracy under attack:        | 20.0%  |
| Attack success rate:          | 78.95% |
| Average perturbed word %:     | 26.23% |
| Average num. words per input: | 8.7    |
| Avg num queries:              | 68.42  |
+-------------------------------+--------+


20 adversarial examples generated.
Starting fine-tuning...
{'loss': 0.028, 'grad_norm': 0.001098243286833167, 'learning_rate': 2.435897435897436e-05, 'epoch': 7.6923076923076925}
{'train_runtime': 19.1234, 'train_samples_per_second': 407.878, 'train_steps_per_second': 50.985, 'train_loss': 0.014351094119632856, 'epoch': 15.0}


[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Fine-tuned model saved to ./defended_model.
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 7 / 13 / 0 / 20: 100%|██████████| 20/20 [00:03<00:00,  6.66it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 7      |
| Number of failed attacks:     | 13     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 65.0%  |
| Attack success rate:          | 35.0%  |
| Average perturbed word %:     | 36.38% |
| Average num. words per input: | 8.7    |
| Avg num queries:              | 86.65  |
+-------------------------------+--------+


### Answer point 2

With more epochs, the fine-tuned model improved its performance, increasing accuracy under attack from 55% to 65%, which is an improvement of 10 percentage points

## Solution 3

In [11]:
# Solution 3

from textattack.attack_recipes import DeepWordBugGao2018

def generate_adversarial_examples_with_deep_word(model_wrapper, dataset, num_examples: int = 20, verbose: bool = True):
    """Generate adversarial examples with TextAttack.

    Returns a list of dicts with keys:
    - `text`: perturbed text (or original text if the attack fails)
    - `label`: ground-truth label from the original example

    Guardrails:
    - TextAttack can emit failed/skipped results; we keep the original text so fine-tuning can proceed.
    """

    from textattack import AttackArgs, Attacker

    attack = DeepWordBugGao2018.build(model_wrapper)
    attack_args = AttackArgs(
        num_examples=num_examples,
        shuffle=True,        # randomize which samples are attacked
        disable_stdout=True, # silence TextAttack internal prints (our prints still show)
    )
    attacker = Attacker(attack, dataset, attack_args)

    results = []
    for i, result in enumerate(attacker.attack_dataset()):
        # TextAttack results differ for success/fail/skip; guard against missing fields.
        try:
            original_text = result.original_text()
        except Exception:
            original_text = None

        try:
            perturbed_text = result.perturbed_text()
        except Exception:
            perturbed_text = None

        original_label = getattr(getattr(result, "original_result", None), "ground_truth_output", None)
        if original_label is None:
            raise ValueError("Missing ground-truth label; check that the TextAttack dataset includes labels.")

        predicted_label = getattr(getattr(result, "perturbed_result", None), "output", None)

        # If the attack fails, fall back to the original text to avoid downstream crashes.
        attack_text = perturbed_text or original_text
        if attack_text is None:
            continue

        if verbose:
            print(f"\nExample {i + 1}")
            print("Original text:", original_text)
            print("Perturbed text:", perturbed_text)
            print("Ground-truth label:", original_label)
            print("Predicted label after attack:", predicted_label)
            print("Attack successful:", original_label != predicted_label)

        results.append({
            "text": attack_text,
            "label": int(original_label),
        })

    return results

In [12]:

def main():
    # Configuration knobs for the Activities section.
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"

    # How many examples to attack (also equals the number of adversarial examples we attempt to generate).
    attack_examples = 20

    # How many clean examples to include for adversarial training.
    clean_train_examples = 500

    # Validation subset size for quick iteration.
    val_examples = 100

    # Fine-tuning epochs (increase for Activity 2).
    num_train_epochs = 5

    print("Loading model and dataset...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

    # GLUE SST-2: binary sentiment classification.
    hf_dataset = load_dataset("glue", "sst2")
    train_raw = hf_dataset["train"]
    val_raw = hf_dataset["validation"]

    # TextAttack expects a list of (text, label) pairs.
    from textattack.datasets import Dataset

    sample_for_attack = list(zip(
        train_raw["sentence"][:attack_examples],
        train_raw["label"][:attack_examples],
    ))
    textattack_dataset = Dataset(sample_for_attack)

    print("Generating adversarial examples with deep word...")
    adv_examples = generate_adversarial_examples_with_deep_word(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )

    print(f"{len(adv_examples)} adversarial examples generated.")

    # Combine clean and adversarial samples for adversarial training.
    clean_data = [
        {"text": x, "label": y}
        for x, y in zip(
            train_raw["sentence"][:clean_train_examples],
            train_raw["label"][:clean_train_examples],
        )
    ]
    combined_data = clean_data + adv_examples

    train_dataset = CustomDataset(combined_data, tokenizer)
    val_dataset = CustomDataset(
        [
            {"text": x, "label": y}
            for x, y in zip(
                val_raw["sentence"][:val_examples],
                val_raw["label"][:val_examples],
            )
        ],
        tokenizer,
    )

    print("Starting fine-tuning...")

    args = TrainingArguments(
        output_dir="./defended_model",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        logging_dir="./logs",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    trainer.train()

    # Persist the defended model locally.
    model.save_pretrained("./defended_model")
    tokenizer.save_pretrained("./defended_model")
    print("Fine-tuned model saved to ./defended_model.")

    # Re-wrap the fine-tuned model and rerun the attack to see whether robustness improved.
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
    _ = generate_adversarial_examples_with_deep_word(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )


if __name__ == "__main__":
    main()


Loading model and dataset...


textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Generating adversarial examples with deep word...
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  CompositeTransformation(
    (0): WordSwapNeighboringCharacterSwap(
        (random_one):  True
      )
    (1): WordSwapRandomCharacterSubstitution(
        (random_one):  True
      )
    (2): WordSwapRandomCharacterDeletion(
        (random_one):  True
      )
    (3): WordSwapRandomCharacterInsertion(
        (random_one):  True
      )
    )
  (constraints): 
    (0): LevenshteinEditDistance(
        (max_edit_distance):  30
        (compare_against_original):  True
      )
    (1): RepeatModification
    (2): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 12 / 7 / 1 / 20: 100%|██████████| 20/20 [00:00<00:00, 24.84it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 12     |
| Number of failed attacks:     | 7      |
| Number of skipped attacks:    | 1      |
| Original accuracy:            | 95.0%  |
| Accuracy under attack:        | 35.0%  |
| Attack success rate:          | 63.16% |
| Average perturbed word %:     | 36.64% |
| Average num. words per input: | 8.7    |
| Avg num queries:              | 16.79  |
+-------------------------------+--------+



Example 1
Original text: with his usual intelligence and subtlety 
Perturbed text: with his usual inteZligence and subtVlety 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 2
Original text: on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 
Perturbed text: on the Qorst revenge-of-the-nerds clicEés the filmmakedrs could redge up 
Ground-truth label: 0
Predicted label after attack: 0
Attack successful: False

Example 3
Original text: remains utterly satisfied to remain the same throughout 
Perturbed text: regains utterly satisfied to remain the same throughout 
Ground-truth label: 0
Predicted label after attack: 1
Attack successful: True

Example 4
Original text: that 's far too tragic to merit such superficial treatment 
Perturbed text: that 's ar too traguc to merti such superficilal treatmeTnt 
Ground-truth label: 0
Predicted label after attack: 0
Attack successful: False

Example 5
Original text: for those moviegoers wh

textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Fine-tuned model saved to ./defended_model.
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  CompositeTransformation(
    (0): WordSwapNeighboringCharacterSwap(
        (random_one):  True
      )
    (1): WordSwapRandomCharacterSubstitution(
        (random_one):  True
      )
    (2): WordSwapRandomCharacterDeletion(
        (random_one):  True
      )
    (3): WordSwapRandomCharacterInsertion(
        (random_one):  True
      )
    )
  (constraints): 
    (0): LevenshteinEditDistance(
        (max_edit_distance):  30
        (compare_against_original):  True
      )
    (1): RepeatModification
    (2): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 8 / 12 / 0 / 20: 100%|██████████| 20/20 [00:01<00:00, 17.16it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 8      |
| Number of failed attacks:     | 12     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 60.0%  |
| Attack success rate:          | 40.0%  |
| Average perturbed word %:     | 45.11% |
| Average num. words per input: | 8.7    |
| Avg num queries:              | 22.95  |
+-------------------------------+--------+

Example 1
Original text: with his usual intelligence and subtlety 
Perturbed text: with his usula iYntelligence and subtHety 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 2
Original text: on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 
Perturbed text: on the worst revenge-of-the-nerds clichés the filmmakres mould ddredge up 
Ground-truth label: 0
Predicted label after 

### Answer point 3

DeepWordBugGao2018 is easier to correct and doesn't have more successful attacks than PWWSRen2019



## Solution 4

In [13]:
# Solution 2
def main():
    # Configuration knobs for the Activities section.
    model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

    # How many examples to attack (also equals the number of adversarial examples we attempt to generate).
    attack_examples = 20

    # How many clean examples to include for adversarial training.
    clean_train_examples = 500

    # Validation subset size for quick iteration.
    val_examples = 100

    # Fine-tuning epochs (increase for Activity 2).
    num_train_epochs = 5

    print("Loading model and dataset...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

    # GLUE SST-2: binary sentiment classification.
    hf_dataset = load_dataset("glue", "sst2")
    train_raw = hf_dataset["train"]
    val_raw = hf_dataset["validation"]

    # TextAttack expects a list of (text, label) pairs.
    from textattack.datasets import Dataset

    sample_for_attack = list(zip(
        train_raw["sentence"][:attack_examples],
        train_raw["label"][:attack_examples],
    ))
    textattack_dataset = Dataset(sample_for_attack)

    print("Generating adversarial examples...")
    adv_examples = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=False,
    )

    print(f"{len(adv_examples)} adversarial examples generated.")

    # Combine clean and adversarial samples for adversarial training.
    clean_data = [
        {"text": x, "label": y}
        for x, y in zip(
            train_raw["sentence"][:clean_train_examples],
            train_raw["label"][:clean_train_examples],
        )
    ]
    combined_data = clean_data + adv_examples

    train_dataset = CustomDataset(combined_data, tokenizer)
    val_dataset = CustomDataset(
        [
            {"text": x, "label": y}
            for x, y in zip(
                val_raw["sentence"][:val_examples],
                val_raw["label"][:val_examples],
            )
        ],
        tokenizer,
    )

    print("Starting fine-tuning...")

    args = TrainingArguments(
        output_dir="./defended_model",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        logging_dir="./logs",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    trainer.train()

    # Persist the defended model locally.
    model.save_pretrained("./defended_model")
    tokenizer.save_pretrained("./defended_model")
    print("Fine-tuned model saved to ./defended_model.")

    # Re-wrap the fine-tuned model and rerun the attack to see whether robustness improved.
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
    _ = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=False,
    )


if __name__ == "__main__":
    main()


Loading model and dataset...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


20 adversarial examples generated.
Starting fine-tuning...
{'train_runtime': 12.2611, 'train_samples_per_second': 212.053, 'train_steps_per_second': 26.507, 'train_loss': 0.21444871168870192, 'epoch': 5.0}


[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
textattack: Unknown if model of class <class 'transformers.models.roberta.modeling_roberta.RobertaForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Fine-tuned model saved to ./defended_model.
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 9 / 11 / 0 / 20: 100%|██████████| 20/20 [00:05<00:00,  3.89it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 9      |
| Number of failed attacks:     | 11     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 55.0%  |
| Attack success rate:          | 45.0%  |
| Average perturbed word %:     | 32.14% |
| Average num. words per input: | 8.7    |
| Avg num queries:              | 85.3   |
+-------------------------------+--------+


## Answer point 4

The new model (Twitter roberta) is less accurate than the bert model. Even the original accuracy is worse (50% compared to the 95% of roberta). However, after training, its original accuracy improves, and its accuracy under attack also improves, equaling the performance of the BERT model. This indicates that we can fine-tune a weaker model and get results pretty similar to more advanced models.